# Cirrhosis patient survival - classification track

23CSE301 Machine Learning capstone. Dataset is the Mayo Clinic PBC (primary biliary cirrhosis) trial data, `data/cirrhosis.csv`, which has 418 patients and 20 columns.



## 2. Setup

In [2]:
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("sklearn", sklearn.__version__)

pandas 2.3.0
numpy 2.0.2
sklearn 1.7.0


## 3. Data loading

In [3]:
df_raw = pd.read_csv("data/cirrhosis.csv")
print("shape:", df_raw.shape)
df_raw.head()

shape: (418, 20)


,ID,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,1,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,2,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,3,1012,D,D-penicillamine,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,4,1925,D,D-penicillamine,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,5,1504,CL,Placebo,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0


Dataset Description

- ID - patient id, not useful so it gets dropped
- N_Days - days from registration until death / transplant / the study analysis date (this one is a problem, see section 5)
- Status - the target. C = censored (alive), CL = censored due to transplant, D = death
- Drug - D-penicillamine or Placebo. NaN means the patient wasn't in the randomised trial
- Age - age in DAYS at registration
- Sex - M / F
- Ascites - fluid build up in the abdomen, Y/N
- Hepatomegaly - enlarged liver, Y/N
- Spiders - spider naevi (vascular skin lesions), Y/N
- Edema - N = none, S = present or resolved with diuretics, Y = present even with diuretics
- Bilirubin - serum bilirubin in mg/dL, goes up when the liver can't excrete properly
- Cholesterol - serum cholesterol mg/dL
- Albumin - serum albumin g/dL, shows synthetic liver function
- Copper - urine copper ug/day, builds up in cholestatic disease
- Alk_Phos - alkaline phosphatase U/litre, bile duct obstruction
- SGOT - SGOT / AST in U/mL, liver cell injury
- Tryglicerides - triglycerides mg/dL (spelt wrong in the dataset)
- Platelets - platelet count per 1000/mL, falls with portal hypertension
- Prothrombin - prothrombin time in seconds, clotting so again synthetic liver function
- Stage - histologic stage of the disease, 1 (early) to 4 (cirrhosis)

In [4]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "non_null": df_raw.notnull().sum(),
    "missing": df_raw.isnull().sum(),
    "missing_%": round(df_raw.isnull().mean() * 100, 1),
    "n_unique": df_raw.nunique()
})
audit

,dtype,non_null,missing,missing_%,n_unique
ID,int64,418,0,0.0,418
N_Days,int64,418,0,0.0,399
Status,object,418,0,0.0,3
Drug,object,312,106,25.4,2
Age,int64,418,0,0.0,344
Sex,object,418,0,0.0,2
Ascites,object,312,106,25.4,2
Hepatomegaly,object,312,106,25.4,2
Spiders,object,312,106,25.4,2
Edema,object,418,0,0.0,3


In [ ]:
# target distribution
print(df_raw["Status"].value_counts())
print()
print(df_raw["Status"].value_counts(normalize=True) * 100)

counts = df_raw["Status"].value_counts()
print()
print("imbalance ratio C : CL =", round(counts["C"] / counts["CL"], 1), ": 1")
print("majority class baseline accuracy =", round(counts.max() / len(df_raw) * 100, 2), "%")

In [ ]:
df_raw.drop(columns=["ID"]).describe().T.round(2)

### audi obervations

418 patients and 20 columns.

There are 3 classes for the target feature Status:

- C = censored, basically still alive at the end of the study (232 patients)
- CL = censored because they got a liver transplant (25 patients)
- D = death (161 patients)

So it's a 3 class classification, and there is a very big class imbalance; CL makes up only  6% of the data.

The missing values are not random completely random; 106 rows are missing all Drug, Ascites, Hepatomegaly, Spiders, Alk_Phos and SGOT together. maybe these are the patients who agreed to basic measurements but never joined the randomised trial.

Cholesterol (32% missing) and Tryglicerides (33%) are the values with the most missing.

Age is stored in days (9598 to 28650, so about 26 to 78 years).